### Data Pre Processing
There are several categorical columns **BUT** the model only understand numbers so we have to convert them to numbers. <br> 
But we cannot let them randomly be 1,2,3 because model might make a pattern like 3>2>1 and be confused. <br><br>
When there is a clear ranking given in categorical columns, we use **Ordinal Encoding**: giving straight numerical values to categorical columns(like Savings and Checking Account in our dataset where a heirarchy is provided) and we are assuming that "No account" field which we add will be lowest in the order. <br><br>
But since, in our dataset there is no clear ranking and spacing, we would be using **One Hot Encoding:** creating a new column for each categorical value and then give 0/1 based on existing/not existing.<br><br>


In [65]:
import pandas as pd 
import numpy as np

In [66]:
df = pd.read_csv('data.csv')
df.head(2)

,Unnamed: 0,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose,Risk
0,0,67,male,2,own,NaN,little,1169,6,radio/TV,good
1,1,22,female,2,own,little,moderate,5951,48,radio/TV,bad


In [67]:
# dropping unnamed column which is at index 0
df.drop(columns = ['Unnamed: 0'], inplace = True) # see only columns so no axis needed
# OR
# df.drop(df.columns[0], axis = 1, inplace = True)  
'''extracting a raw value df.columns[0] and pandas need an axis 
to look for it'''

'extracting a raw value df.columns[0] and pandas need an axis \nto look for it'

In [68]:
# dropping the target column
X = df.drop(columns = ['Risk'])
y = df['Risk']

In [69]:
y = y.map({'bad': 1, 'good': 0}) # map instead of ordinal encoding because we decide which is 1 and which is 0

In [70]:
# Fill missing values
X["Saving accounts"] = X["Saving accounts"].fillna("No Account")
X["Checking account"] = X["Checking account"].fillna("No Account")

There are shortcuts for preprocessing like pd.get_dummies(df) but we would create a proper pipeline using ColumnTransfer and Pipeline 

In [71]:
dff = pd.get_dummies(X) # this is a shortcut for one hot encoding
dff.head(1)

,Age,Job,Credit amount,Duration,Sex_female,Sex_male,Housing_free,Housing_own,Housing_rent,Saving accounts_No Account,...,Checking account_moderate,Checking account_rich,Purpose_business,Purpose_car,Purpose_domestic appliances,Purpose_education,Purpose_furniture/equipment,Purpose_radio/TV,Purpose_repairs,Purpose_vacation/others
0,67,2,1169,6,False,True,False,True,False,True,...,False,False,False,False,False,False,False,True,False,False


**Now there is one more column left to talk about, Job, which is surprisingly numerical.**<br><br>
**Check what values job holds, comes out to be 0,1,2,3 which COULD possibly be another sequence like 0=unskilled, 1=semi-skilled and so on.** <br><br>
**This means, job column is by default ordinal encoded, IT IS A GUESS NOT A CONCRETE DATA ATTRIBUTE**

In [72]:
'''value counts give the frequency of each value
sort_index sorts the output based on the value rather than frequency.'''
X["Job"].value_counts().sort_index()

Job
0     22
1    200
2    630
3    148
Name: count, dtype: int64

<hr>

In [73]:
# TRAIN-TEST SPLIT 
from sklearn.model_selection import train_test_split

# stratify = y the train and test both data will have same ratio of the target variable values
# if DF has 70 good and 30 bad then train and test will have 70:30 ratio too
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(800, 9)
(200, 9)


### Transformations
**After train-test split: next step is applying the transformations we discussed for all columns** <br><br>
**ColumnTransfer** allows different pre processing steps to be applied to different columns <br>
It is different from a **Pipeline** (transformations one after another) as it provides parallel transformations.<br><br>
ColumnTransformer class's constructor takes a list of tuples transformers = [(), (), ()] as argument and each tuple contains a transformer<br>
AND each transformer will have (name, transformer, list of columns to apply) <br><br>
StandardScaler(copy, with_mean, with_std) <br>
OrdinalEncoder(categories=[]), categories is a list of lists, each list contains the ORDER to be used for each column in the columns list <br>
OneHotEncoder(handle_unknowns = "ignore"), ignores any new value in the test data which wasn't there in training data

In [74]:
from sklearn.compose import ColumnTransformer # for applying multiple transformations to multiple columns at once
from sklearn.preprocessing import ( # importing all the transformers
    OneHotEncoder, 
    OrdinalEncoder,
    StandardScaler # for numerical features (study about min-max and standardScaler)
)

# making feature groups to apply different transformations to them 
numerical_features = [
    "Age",
    "Credit amount",
    "Duration"
]

ordinal_features = [
    "Job",
    "Saving accounts",
    "Checking account"
]

nominal_features = [ # for one hot encoding
    "Sex",
    "Housing",
    "Purpose"
]

In [75]:
job_order = [0, 1, 2, 3]

saving_order = [
    "No Account",
    "little",
    "moderate",
    "rich",
    "quite rich"
]

checking_order = [
    "No Account",
    "little",
    "moderate",
    "rich"
]

In [86]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "ord",
            OrdinalEncoder(
                categories=[
                    job_order, # used for jobs
                    saving_order, # used for saving accounts
                    checking_order # used for checking accounts
                ]
            ),
            ordinal_features # contains job, saving accounts and checking accounts
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            nominal_features
        )
    ]
)

**After creating a processor, we need to transform the train test data we created** <br><br>
fit() -> learns from the data, different transformers(encoders, scalers, etc) learn different things <br>
transform() -> applies transformation which our processor holds <br>
fit_transform() -> learns from the data AND applies transformations <br><br>
**#Note**: fit is only used in train data BECAUSE the preprocessor learns from train data, transforms in a certain way and model is trained.<br>
If processor learns AGAIN from the test data, it will transform it in a certain way and model won't be able to predict correctly. <br> 
**The preprocessor should learn from train data only and transform both train and test data with that knowledge only**<br><br>
The output of transform() is a **feature matrix** numpy array -> array([[row1], [row2], ..])<br>
To convert matrix in DF, preprocessor.get_features_names_out() and then pass columns and matrix in pd.DataFrame()<br>

In [87]:

X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

In [91]:
X_train_final = pd.DataFrame(X_train_transformed, columns = preprocessor.get_feature_names_out())
X_test_final = pd.DataFrame(X_test_transformed, columns = preprocessor.get_feature_names_out())
X_train_final.head(2)
X_test_final.head(2)

,num__Age,num__Credit amount,num__Duration,ord__Job,ord__Saving accounts,ord__Checking account,cat__Sex_female,cat__Sex_male,cat__Housing_free,cat__Housing_own,cat__Housing_rent,cat__Purpose_business,cat__Purpose_car,cat__Purpose_domestic appliances,cat__Purpose_education,cat__Purpose_furniture/equipment,cat__Purpose_radio/TV,cat__Purpose_repairs,cat__Purpose_vacation/others
0,0.061263,-0.477788,-0.234548,2.0,3.0,2.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,-0.119922,-0.497625,-0.742595,3.0,1.0,2.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
